# 00 — Setup and frozen analysis plan

First stage of the sequential Colab pipeline (Methods 2.13). Freezes the analysis plan and random seed used by every downstream notebook, before any data are touched.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/<your-repo>.git"  # TODO: set to the published GitHub repo URL
REPO_DIR_NAME = "pcos_network_architecture_jcem_v3_jcem_10of10"


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    PROJECT_ROOT = Path("/content") / REPO_DIR_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not ((PROJECT_ROOT / "data_raw").exists() and (PROJECT_ROOT / "outputs").exists()):
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            break
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json

plan = {
    'title': 'Endocrine-metabolic network architecture reveals key bridge biomarkers in polycystic ovary syndrome',
    'target_journal': 'JCEM',
    'version': 'v2_methodologically_revised',
    'primary_improvements': [
        'primary network uses only directly measured biomarkers',
        'derived ratios excluded from primary network and moved to sensitivity',
        'Graphical LASSO alpha selected by cross-validation',
        'bootstrap stability increased to 500 resamples',
        'secondary full clinical network retained as sensitivity',
    ],
    'random_seed': SEED,
}

out_dir = PROJECT_ROOT / 'outputs' / '00_setup'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'analysis_plan_frozen_v2.json').write_text(json.dumps(plan, indent=2), encoding='utf-8')
print((out_dir / 'analysis_plan_frozen_v2.json').read_text())
